
# Generate an EDA Summary Helper
In the previous lesson, you built a generic helper.In this notebook we sharpen that helper for a specific job: **EDA or exploratory data analysis** such as descriptive statistics, missing values, and distributions.

A High Level diagram of the helper
![](../../images/eda_helper_clean_flow.png)



## 1 — Setup and Imports

The completed notebook includes saved outputs so you can review the expected result without my local `.env` file. To rerun the Gemini cells, create your own `.env` file in the project root with `GEMINI_API_KEY=your-gemini-api-key-here`.


In [ ]:
%pip install -qq google-genai pandas scikit-learn matplotlib seaborn python-dotenv


In [ ]:
import os
import re
from google import genai
from dotenv import load_dotenv
import pandas as pd

import warnings

warnings.filterwarnings("ignore")

# Load environment variables from a .env file into the environment
load_dotenv()

# Read the Gemini API key from the environment variables
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

# Create a Gemini client using the API key
# This client will be used to send requests to Gemini models
client = genai.Client(api_key=GEMINI_API_KEY)


### Load the dataset

In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


df = pd.read_csv("../../data/hr_analytics.csv")
df.head()

## 2 — Context Matters

We’ll now update the helper function from previous video with more context. Observe how the output improves as we provide the model with more context about the dataset.


In [ ]:
# System prompt that tells the model exactly what kind of code it is allowed to return.
# We restrict it to pandas code that operates on the existing DataFrame `df` and
# stores the final output in `result_df`, with no imports or extra text.

EDA_SUMMARY_PROMPT = (
    "You are an EDA assistant. "
    "Write pandas code using the existing DataFrame variable `df`. "
    "Use up-to-date pandas 2.x and Python 3.10+ syntax. Avoid deprecated arguments or methods. "
    "Focus on descriptive statistics: shape, dtypes, missing values, "
    "unique counts, central tendency and spread. "
    "Use only valid pandas operations and function names. "
    "Do not invent custom aggregation names or shorthand labels inside pandas methods. "
    "If you need quartiles or similar statistics, compute them explicitly with valid pandas code such as quantile(). "
    "Store the final result in `result_df` as a DataFrame and not multiple variables."
    "Do NOT create a new DataFrame from scratch. "
    "Do NOT include import statements. "
    "Return ONLY executable Python code, no explanations."
)


In [ ]:
QUESTION = "Identify any columns with mixed or inconsistent data formats"

#### v1 — Column names + dtypes

In [ ]:
context_v1 = f"Columns: {list(df.columns)}"

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"{context_v1}\n\nQuestion: {QUESTION}",  # Makes the output deterministic
    config={
        "temperature": 0.0,
        "seed": 42,
        "system_instruction": EDA_SUMMARY_PROMPT,
    },
)


In [ ]:
# Extract executable Python from the model response.
response.text

In [ ]:
text = (response.text or "").strip()
match = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
code = match.group(1).strip() if match else text


In [ ]:
code


In [ ]:
# Define a restricted execution environment.
# The generated code can only access pandas (pd) and the DataFrame (df).
env = {"pd": pd, "df": df.copy()}

# Execute the generated code in the restricted environment.
exec(code, env, env)

# The model is instructed to store the final output in `result_df`.
env.get("result_df")


Using `exec()` to run LLM-generated code is powerful but requires care. In a production environment you'd want to add error handling (try/except around the exec), validate the output type, and potentially sandbox the execution. For learning purposes, the pattern here is fine.

#### v2 — Column names + dtypes + sample rows

In [ ]:
context_v2 = (
    f"Columns: {list(df.columns)}\n"
    f"Dtypes:\n{df.dtypes.to_string()}\n"
    f"Sample (10 rows):\n{df.head(10).to_string()}"
)


In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=f"{context_v2}\n\nQuestion: {QUESTION}",
    config={
        "temperature": 0.0,
        "seed": 42,
        "system_instruction": EDA_SUMMARY_PROMPT,
    },
)


In [ ]:
text = (response.text or "").strip()
match = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE)
code = match.group(1).strip() if match else text
env = {"pd": pd, "df": df.copy()}
exec(code, env, env)
env.get("result_df")


Sample rows allow the model to see actual values, which helps it detect issues like mixed formats, encoded categories, or unexpected string patterns.

## 3 - The EDA Summary Helper

In [ ]:
def eda_summary_helper(question, frame, show_code=False):
    """Ask a question about a DataFrame; get back a DataFrame.

    Args:
        question:      Plain-English EDA question.
        frame:         Input DataFrame. Not modified in place.
        show_code:     If True, print generated code before executing.

    Returns:
        result DataFrame.
    """
    prompt = (
        f"Columns: {list(frame.columns)}"
        f"Dtypes:{frame.dtypes.to_string()}"
        f"Sample (10 rows):{frame.head(10).to_string()}"
        f"Question: {question}"
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config={
            "temperature": 0.0,  # Makes the output deterministic
            "seed": 42,
            "system_instruction": EDA_SUMMARY_PROMPT,  # System level instructions that define rules or behavior for the model
        },
    )

    text = (response.text or "").strip()
    match = re.search(
        r"```(?:python)?\s*(.*?)```", text, flags=re.DOTALL | re.IGNORECASE
    )
    code = match.group(1).strip() if match else text

    if show_code:
        print("--- Generated Code ---")
        print(code)
        print("---")

    # Define a restricted execution environment.
    # The generated code can only access pandas (pd) and the DataFrame (df).
    env = {"pd": pd, "df": frame.copy()}

    # Execute the generated code in the restricted environment.
    exec(code, env, env)

    # The model is instructed to store the final output in `result_df`.
    result = env.get("result_df")
    if result is None:
        raise RuntimeError("Generated code did not assign `result_df`.")
    return result

In [ ]:
result_df = eda_summary_helper(
    "Give me a summary of the DataFrame with descriptive statistics", df, show_code=True
)
result_df

## 4 - Save for Reuse



In [ ]:
import inspect

components = [
    "import os",
    "import re",
    "import pandas as pd",
    "from google import genai",
    "from dotenv import load_dotenv",
    "",
    "load_dotenv()",
    "client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))",
    "",
    f"EDA_SUMMARY_PROMPT = {repr(EDA_SUMMARY_PROMPT)}",
    "",
    inspect.getsource(eda_summary_helper),
]

with open("eda_summary_helper.py", "w", encoding="utf-8") as f:
    f.write("\n".join(components))

print("Saved eda_summary_helper.py")
